In [1]:
import os
import sys
sys.path.append("../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from torch.utils.data import DataLoader
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data,
)
from utils.model_utils.save_module import save_module
from utils.model_utils.load_model import load_model
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.perturb import make_example
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune_head import head_importance_prunning
from utils.prune_utils.prune import prune_concern_identification

In [3]:
name= "bert-tiny-yahoo"
device = torch.device("cuda:0")
checkpoint = None
batch_size=16
num_workers=4
num_samples = 128
head_pruning_ratio = 0.5
ci_ratio = 0.5
seed = 44
include_layers = ["intermediate", "output"]
exclude_layers = ["attention"]

In [4]:
script_start_time = datetime.now()
print(f"Script started at: {script_start_time.strftime('%Y-%m-%d %H:%M:%S')}")

Script started at: 2024-09-13 15:53:45


In [5]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.
{'model_name': 'Models/bert-tiny-yahoo', 'tokenizer_name': 'sadickam/sdg-classification-bert', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}
The model Models/bert-tiny-yahoo is loaded.


In [6]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    model_config.dataset_name, batch_size=batch_size, num_workers=num_workers, do_cache=True, seed=seed
)

Loading cached dataset YahooAnswersTopics.
train.pkl is loaded from cache.
valid.pkl is loaded from cache.
test.pkl is loaded from cache.
The dataset YahooAnswersTopics is loaded
{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/YahooAnswersTopics', 'task_type': 'classification'}


In [7]:
# positive_embeddings, negative_embeddings= make_example(
#     model,
#     model_config,
#     data_loader=valid_dataloader,
#     example_num=3000,
#     top_emb=3,
#     true_ratio=0.5,
#     step_eps=0.01,
#     max_eps=10.0
# )

In [8]:
# from utils.dataset_utils.load_dataset import save_cache, load_from_cache
# save_cache(positive_embeddings, "./", "positive_embeddings.pkl")
# save_cache(negative_embeddings, "./", "negative_embeddings.pkl")

In [9]:
from utils.dataset_utils.load_dataset import save_cache, load_from_cache
positive_embeddings = load_from_cache("./", "positive_embeddings.pkl")
negative_embeddings = load_from_cache("./", "negative_embeddings.pkl")

positive_embeddings.pkl is loaded from cache.
negative_embeddings.pkl is loaded from cache.


In [10]:
pos_dataloader = DataLoader(positive_embeddings, batch_size=batch_size, shuffle=True, num_workers=0)
neg_dataloader = DataLoader(negative_embeddings, batch_size=batch_size, shuffle=True, num_workers=0)

In [11]:
# print("Evaluate the original model")
# result = evaluate_model(model, model_config, test_dataloader)

In [12]:
for concern in range(num_labels):
    valid = copy.deepcopy(valid_dataloader)
    
    module = copy.deepcopy(model)
    pos = copy.deepcopy(pos_dataloader)
    neg = copy.deepcopy(neg_dataloader)

    positive_samples = SamplingDataset(
        pos, concern, num_samples, num_labels, True, 4, device=device, resample=False, seed=seed
    )
    negative_samples = SamplingDataset(
        neg, concern, num_samples, num_labels, False, 4, device=device, resample=False, seed=seed
    )
    all_samples = SamplingDataset(
        pos, 200, num_samples, num_labels, False, 4, device=device, resample=False, seed=seed
    )

    head_importance_prunning(
        module, model_config, positive_samples, head_pruning_ratio
    )
    
    prune_concern_identification(
        module,
        model_config,
        positive_samples,
        negative_samples,
        include_layers=include_layers,
        exclude_layers=exclude_layers,
        sparsity_ratio=ci_ratio,
    )
        
    print(f"Evaluate the pruned model {concern}")
    result = evaluate_model(module, model_config, test_dataloader)
    get_sparsity(module)

    similar(model, module, valid, concern, num_samples, num_labels, device=device, seed=seed)

    save_module(module, "Modules/", f"head_pruned_ci_{name}_{head_pruning_ratio}p_{ci_ratio}p_class{concern}")

Total heads to prune: 2


BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


{(0, 1), (1, 1)}
Evaluate the pruned model 0


Evaluating the model:   0%|                                                                    | 0/1875 [00:00…

Loss: 1.3521
Precision: 0.6335, Recall: 0.5919, F1-Score: 0.5975
              precision    recall  f1-score   support

           0       0.54      0.48      0.51      2941
           1       0.71      0.49      0.58      2997
           2       0.61      0.68      0.64      3016
           3       0.30      0.64      0.41      2978
           4       0.77      0.75      0.76      3017
           5       0.68      0.78      0.73      3004
           6       0.63      0.39      0.48      3037
           7       0.73      0.46      0.56      3026
           8       0.66      0.60      0.63      2997
           9       0.70      0.65      0.68      2987

    accuracy                           0.59     30000
   macro avg       0.63      0.59      0.60     30000
weighted avg       0.63      0.59      0.60     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

...

Loss: 1.3027
Precision: 0.6323, Recall: 0.5951, F1-Score: 0.5931
              precision    recall  f1-score   support

           0       0.52      0.47      0.49      2941
           1       0.74      0.44      0.55      2997
           2       0.67      0.62      0.64      3016
           3       0.32      0.62      0.43      2978
           4       0.67      0.81      0.73      3017
           5       0.65      0.83      0.73      3004
           6       0.75      0.30      0.43      3037
           7       0.67      0.53      0.59      3026
           8       0.64      0.65      0.65      2997
           9       0.69      0.68      0.68      2987

    accuracy                           0.60     30000
   macro avg       0.63      0.60      0.59     30000
weighted avg       0.63      0.60      0.59     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|                                                                    | 0/1875 [00:00…

Loss: 1.4348
Precision: 0.6492, Recall: 0.5411, F1-Score: 0.5422
              precision    recall  f1-score   support

           0       0.34      0.60      0.43      2941
           1       0.78      0.29      0.42      2997
           2       0.82      0.38      0.52      3016
           3       0.28      0.66      0.40      2978
           4       0.77      0.62      0.69      3017
           5       0.73      0.79      0.76      3004
           6       0.83      0.21      0.34      3037
           7       0.56      0.64      0.60      3026
           8       0.75      0.50      0.60      2997
           9       0.63      0.72      0.67      2987

    accuracy                           0.54     30000
   macro avg       0.65      0.54      0.54     30000
weighted avg       0.65      0.54      0.54     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|                                                                    | 0/1875 [00:00…

Loss: 1.3528
Precision: 0.6318, Recall: 0.5921, F1-Score: 0.5975
              precision    recall  f1-score   support

           0       0.53      0.48      0.51      2941
           1       0.71      0.49      0.58      2997
           2       0.61      0.68      0.64      3016
           3       0.30      0.63      0.41      2978
           4       0.76      0.75      0.76      3017
           5       0.68      0.78      0.73      3004
           6       0.64      0.39      0.48      3037
           7       0.72      0.46      0.56      3026
           8       0.66      0.60      0.63      2997
           9       0.70      0.65      0.68      2987

    accuracy                           0.59     30000
   macro avg       0.63      0.59      0.60     30000
weighted avg       0.63      0.59      0.60     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa